# ROC-AUC и PR-AUC для CatBoost road-only 3.5

Ноутбук строит кривые для финального baseline абсолютного риска: `CatBoost road-only 3.5`.

Используется только test-period (`catboost_road_only_3_5_train_window == False`), чтобы графики соответствовали метрикам из `EXPERIMENTS.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PREDICTIONS_PATH = PROJECT_ROOT / "parquet_output" / "h3_month_catboost_road_only_3_5_predictions.parquet"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "target_had_accident_next_month"
SCORE = "catboost_road_only_3_5_score"
TRAIN_WINDOW = "catboost_road_only_3_5_train_window"

df = pd.read_parquet(PREDICTIONS_PATH)
test = df[~df[TRAIN_WINDOW]].copy()

y_true = test[TARGET].astype(int)
y_score = test[SCORE]

roc_auc = roc_auc_score(y_true, y_score)
pr_auc = average_precision_score(y_true, y_score)
positive_rate = y_true.mean()

print(f"Rows: {len(test):,}")
print(f"Positive rate: {positive_rate:.4%}")
print(f"ROC-AUC: {roc_auc:.6f}")
print(f"PR-AUC: {pr_auc:.6f}")

## ROC curve

ROC-кривая показывает баланс между долей найденных ДТП-строк (`True Positive Rate`) и долей ложных тревог среди отрицательных строк (`False Positive Rate`) при разных порогах риска.

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)

plt.figure(figsize=(7.5, 5.5), dpi=140)
plt.plot(fpr, tpr, linewidth=2.2, label=f"CatBoost road-only 3.5, ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.4, color="#8a8f98", label="Случайное ранжирование")
plt.title("ROC curve: CatBoost road-only 3.5")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.legend(loc="lower right")
plt.tight_layout()

roc_path = REPORTS_DIR / "catboost_road_only_3_5_roc_curve.png"
plt.savefig(roc_path, bbox_inches="tight")
plt.show()

print(roc_path)

## Precision-Recall curve

Precision-Recall кривая важнее для этой задачи, потому что ДТП-строки редкие. Она показывает, как меняется точность положительных срабатываний (`Precision`) при разной полноте найденных ДТП (`Recall`).

In [ ]:
precision, recall, _ = precision_recall_curve(y_true, y_score)

plt.figure(figsize=(7.5, 5.5), dpi=140)
plt.plot(recall, precision, linewidth=2.2, label=f"CatBoost road-only 3.5, PR-AUC = {pr_auc:.3f}")
plt.axhline(positive_rate, linestyle="--", linewidth=1.4, color="#8a8f98", label=f"Базовая частота ДТП = {positive_rate:.2%}")
plt.title("Precision-Recall curve: CatBoost road-only 3.5")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.legend(loc="upper right")
plt.tight_layout()

pr_path = REPORTS_DIR / "catboost_road_only_3_5_pr_curve.png"
plt.savefig(pr_path, bbox_inches="tight")
plt.show()

print(pr_path)

## Короткая трактовка

- `ROC-AUC` показывает общее качество сортировки положительных и отрицательных строк.
- `PR-AUC` показывает качество модели именно для редкого положительного класса и поэтому лучше отражает задачу ДТП.
- Серая пунктирная линия на PR-графике — базовая частота ДТП в test-period. Чем выше кривая над этой линией, тем лучше модель концентрирует будущие ДТП в верхней части скоринга.